# LAPSH: Layer-Adaptive PPO Steering for Hallucination on GSM8K

**Method:** Train a small Policy MLP with PPO on frozen Llama-3.2-3B to learn which layer to steer + how much (magnitude α), using CAA steering vectors computed offline. Reward = answer match + NLI entailment.

**Hardware:** A100 recommended. T4 works with batch_size=4.

## Cell 1: Install Dependencies

In [ ]:
# %%capture
# !pip install -q numpy==1.26.4
# !pip install -q --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install -q --force-reinstall transformers==4.45.0 bitsandbytes==0.43.3 accelerate==0.33.0 datasets==2.21.0 sentence-transformers==3.0.1
# print("Done!")

# %%capture
# !pip install -q numpy==1.26.4
# !pip install -q triton
# !pip install -q --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install -q transformers==4.45.0 bitsandbytes==0.43.3 accelerate==0.33.0 datasets==2.21.0 sentence-transformers==3.0.1
# print("Done!")

%%capture
# Don't touch torch/numpy — use what Colab has
!pip install -q transformers datasets sentence-transformers accelerate
!pip install -q bitsandbytes==0.42.0
print("Done!")

## Cell 2: Imports & Config

In [ ]:
import torch
# Force all custom modules to float32 — avoids Half/Float dtype wars with 4-bit model
torch.set_default_dtype(torch.float32)
import torch.nn as nn, torch.nn.functional as F
import numpy as np, re, json, random
from collections import defaultdict, Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from sentence_transformers import CrossEncoder

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
CANDIDATE_LAYERS  = [8, 12, 16, 20, 24]
N_LAYERS          = len(CANDIDATE_LAYERS)
CAA_SAMPLES       = 500
PPO_TRAIN_SAMPLES = 1000
PPO_EPOCHS, BATCH_SIZE, LR = 3, 8, 3e-4
CLIP_EPS, ENTROPY_COEF, VALUE_COEF, KL_COEF = 0.2, 0.01, 0.5, 0.1
LAMBDA_MATCH, LAMBDA_NLI = 1.0, 0.3
MAX_NEW_TOKENS, ALPHA_SCALE = 256, 20.0
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f"Device: {DEVICE} | Layers: {CANDIDATE_LAYERS}")

## Cell 3: HuggingFace Login

In [ ]:
from huggingface_hub import login
HF_TOKEN = "<REDACTED>"
login(token=HF_TOKEN)

## Cell 4: Load Model (4-bit, frozen)

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
    device_map="auto", token=HF_TOKEN)
model.eval()
for p in model.parameters(): p.requires_grad = False
HIDDEN_SIZE = model.config.hidden_size
print(f"Hidden size: {HIDDEN_SIZE} | Params: {sum(p.numel() for p in model.parameters()):,} (frozen)")

## Cell 5: Load GSM8K

In [ ]:
gsm8k = load_dataset("gsm8k", "main")
train_data = list(gsm8k["train"]); random.shuffle(train_data)
test_data  = list(gsm8k["test"])
caa_data  = train_data[:CAA_SAMPLES]
ppo_data  = train_data[CAA_SAMPLES:CAA_SAMPLES+PPO_TRAIN_SAMPLES]
eval_data = test_data[:200]
print(f"CAA:{len(caa_data)} PPO:{len(ppo_data)} Eval:{len(eval_data)}")

## Cell 6: Utilities

In [ ]:
def extract_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    if m: return m.group(1).replace(',','').strip()
    ns = re.findall(r'\b\d+(?:\.\d+)?\b', text)
    return ns[-1] if ns else None

def answers_match(pred, gold):
    p, g = extract_answer(pred), extract_answer(gold)
    if p is None or g is None: return 0.0
    try: return 1.0 if float(p)==float(g) else 0.0
    except: return 1.0 if p==g else 0.0

def format_prompt(q):
    return (f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"Solve step by step.\n\n{q}"
            f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n")

def get_residual_streams(input_ids, layers):
    cap, hooks = {}, []
    for l in layers:
        def hfn(m,i,o,l=l): cap[l]=o[0].detach().float()
        hooks.append(model.model.layers[l].register_forward_hook(hfn))
    with torch.no_grad(): model(input_ids=input_ids)
    for h in hooks: h.remove()
    return cap

def generate_with_steering(input_ids, svecs, layer, alpha):
    sv = svecs[layer].to(DEVICE)
    def shook(m,i,o): return (o[0]+alpha*sv.unsqueeze(0).unsqueeze(0),)+o[1:]
    h = model.model.layers[layer].register_forward_hook(shook)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    h.remove()
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

print("Utilities defined")

## Cell 7: Compute CAA Steering Vectors

In [ ]:
def compute_caa_vectors(data, layers, n=500):
    pos = {l:[] for l in layers}; neg = {l:[] for l in layers}
    for s in tqdm(data[:n]):
        q, ga = s['question'], s['answer']
        # Positive: gold solution teacher-forced
        pid = tokenizer(format_prompt(q)+ga, return_tensors='pt',
                        truncation=True, max_length=512).input_ids.to(DEVICE)
        ps = get_residual_streams(pid, layers)
        # Negative: high-temp generation
        nid = tokenizer(format_prompt(q), return_tensors='pt',
                        truncation=True, max_length=256).input_ids.to(DEVICE)
        with torch.no_grad():
            gid = model.generate(nid, max_new_tokens=128, do_sample=True,
                                 temperature=1.2, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
        gt = tokenizer.decode(gid[0][nid.shape[1]:], skip_special_tokens=True)
        if answers_match(gt, ga): continue  # skip if model was correct
        nfid = tokenizer(format_prompt(q)+gt, return_tensors='pt',
                         truncation=True, max_length=512).input_ids.to(DEVICE)
        ns = get_residual_streams(nfid, layers)
        for l in layers:
            pos[l].append(ps[l][0].mean(0).cpu())
            neg[l].append(ns[l][0].mean(0).cpu())
    svecs = {}
    for l in layers:
        if not pos[l]: svecs[l]=torch.zeros(HIDDEN_SIZE); continue
        sv = torch.stack(pos[l]).mean(0) - torch.stack(neg[l]).mean(0)
        sv = sv / (sv.norm()+1e-8); svecs[l] = sv
        print(f"  Layer {l:2d}: norm={sv.norm():.4f} pairs={len(pos[l])}")
    return svecs

steering_vectors = compute_caa_vectors(caa_data, CANDIDATE_LAYERS, CAA_SAMPLES)
print("CAA vectors done")

## Cell 8: Policy & Value Networks

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, hs, nl, scale=20.0):
        super().__init__(); self.scale=scale
        self.shared = nn.Sequential(
            nn.Linear(hs*nl,512), nn.LayerNorm(512), nn.ReLU(),
            nn.Linear(512,256), nn.LayerNorm(256), nn.ReLU())
        self.lh = nn.Linear(256, nl)
        self.ah = nn.Linear(256, 1)
    def forward(self, x):
        h = self.shared(x)
        return self.lh(h), torch.tanh(self.ah(h)).squeeze(-1)*self.scale

class ValueNet(nn.Module):
    def __init__(self, hs, nl):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hs*nl,512), nn.LayerNorm(512), nn.ReLU(),
            nn.Linear(512,256), nn.ReLU(), nn.Linear(256,1))
    def forward(self, x): return self.net(x).squeeze(-1)

# policy    = PolicyNet(HIDDEN_SIZE, N_LAYERS, ALPHA_SCALE).to(DEVICE)
# value_net = ValueNet(HIDDEN_SIZE, N_LAYERS).to(DEVICE)
policy    = PolicyNet(HIDDEN_SIZE, N_LAYERS, ALPHA_SCALE).to(torch.float32).to(DEVICE)
value_net = ValueNet(HIDDEN_SIZE, N_LAYERS).to(torch.float32).to(DEVICE)
optimizer = torch.optim.Adam(list(policy.parameters())+list(value_net.parameters()), lr=LR)
print(f"Policy: {sum(p.numel() for p in policy.parameters()):,} params")
print(f"Value:  {sum(p.numel() for p in value_net.parameters()):,} params")

## Cell 9: NLI Reward

In [ ]:
nli = CrossEncoder('cross-encoder/nli-deberta-v3-small', device=DEVICE, max_length=512)

def nli_score(gen, gold):
    return float(nli.predict([(gold[:400], gen[:400])], apply_softmax=True)[0][2])

def reward_fn(gen, gold):
    m = answers_match(gen, gold)
    n = nli_score(gen, gold)
    return LAMBDA_MATCH*m + LAMBDA_NLI*n, m, n

print("NLI reward ready")

## Cell 10: State Extraction

In [ ]:
# def extract_state(question):
#     ids = tokenizer(format_prompt(question), return_tensors='pt',
#                     truncation=True, max_length=256).input_ids.to(DEVICE)
#     streams = get_residual_streams(ids, CANDIDATE_LAYERS)
#     return torch.cat([streams[l][0].mean(0).float() for l in CANDIDATE_LAYERS])
def extract_state(question):
    ids = tokenizer(format_prompt(question), return_tensors='pt',
                    truncation=True, max_length=256).input_ids.to(DEVICE)
    streams = get_residual_streams(ids, CANDIDATE_LAYERS)
    return torch.cat([streams[l][0].mean(0).float() for l in CANDIDATE_LAYERS])
s = extract_state(ppo_data[0]['question'])
print(f"State dim: {s.shape[0]} ({HIDDEN_SIZE} x {N_LAYERS}) OK")

## Cell 11: Rollout & PPO Update

In [ ]:
def collect_rollouts(batch):
    states,lidx,lprobs,vals,rews = [],[],[],[],[]
    matches,nlis,clayers,alphas = [],[],[],[]
    policy.eval(); value_net.eval()
    with torch.no_grad():
        for s in batch:
            st  = extract_state(s['question']); sb = st.unsqueeze(0)
            lg, al = policy(sb)
            probs = F.softmax(lg, -1)
            dist  = torch.distributions.Categorical(probs=probs)
            idx   = dist.sample()
            lp    = dist.log_prob(idx)
            v     = value_net(sb)[0]
            layer = CANDIDATE_LAYERS[idx.item()]; av = al[0].item()
            ids   = tokenizer(format_prompt(s['question']), return_tensors='pt',
                              truncation=True, max_length=256).input_ids.to(DEVICE)
            gen   = generate_with_steering(ids, steering_vectors, layer, av)
            r,m,n = reward_fn(gen, s['answer'])
            states.append(st); lidx.append(idx[0]); lprobs.append(lp[0])
            vals.append(v); rews.append(r); matches.append(m)
            nlis.append(n); clayers.append(layer); alphas.append(av)
    rT = torch.tensor(rews, dtype=torch.float32, device=DEVICE)
    return dict(states=torch.stack(states).to(DEVICE),
                lidx=torch.stack(lidx).to(DEVICE),
                lprobs=torch.stack(lprobs).to(DEVICE),
                vals=torch.stack(vals).to(DEVICE),
                rews=rT, matches=matches, nlis=nlis, clayers=clayers, alphas=alphas)

def ppo_step(ro):
    policy.train(); value_net.train()
    s,idx,old_lp,ret = ro['states'],ro['lidx'],ro['lprobs'].detach(),ro['rews']
    adv = ret - ro['vals'].detach()
    adv = (adv-adv.mean())/(adv.std()+1e-8)
    lg,_ = policy(s); probs=F.softmax(lg,-1); dist=torch.distributions.Categorical(probs=probs)
    nlp = dist.log_prob(idx); ent = dist.entropy().mean()
    ratio = torch.exp(nlp-old_lp)
    pl = -torch.min(ratio*adv, torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*adv).mean()
    vl = F.mse_loss(value_net(s), ret)
    kl = (ratio-1-torch.log(ratio+1e-8)).mean()
    loss = pl + VALUE_COEF*vl - ENTROPY_COEF*ent + KL_COEF*kl
    optimizer.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(list(policy.parameters())+list(value_net.parameters()), 1.0)
    optimizer.step()
    return loss.item(), pl.item(), vl.item(), ent.item()

print("Rollout and PPO functions defined")

## Cell 12: Training Loop

In [ ]:
hist = defaultdict(list)

for epoch in range(PPO_EPOCHS):
    random.shuffle(ppo_data)
    er,em,en,el,eL = [],[],[],[],[]
    nb = len(ppo_data)//BATCH_SIZE
    for bi in tqdm(range(nb), desc=f"Epoch {epoch+1}/{PPO_EPOCHS}"):
        batch = ppo_data[bi*BATCH_SIZE:(bi+1)*BATCH_SIZE]
        ro = collect_rollouts(batch)
        loss,pl,vl,ent = ppo_step(ro)
        er.extend(ro['rews'].cpu().tolist()); em.extend(ro['matches'])
        en.extend(ro['nlis']); el.extend(ro['clayers']); eL.append(loss)
        if (bi+1)%10==0:
            print(f"  [E{epoch+1} B{bi+1}/{nb}] rew={np.mean(er[-80:]):.3f} "
                  f"match={np.mean(em[-80:]):.3f} loss={loss:.4f} ent={ent:.3f}")
    lc = Counter(el)
    for k,v in zip(['reward','match','nli','loss'],[er,em,en,eL]):
        hist[k].append(np.mean(v))
    print(f"Epoch {epoch+1} | rew={np.mean(er):.4f} match={np.mean(em):.4f} | layers={dict(sorted(lc.items()))}")

print("Training done!")

## Cell 13: Training Curves

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12,8))
fig.suptitle('LAPSH Training (GSM8K)', fontsize=14, fontweight='bold')
ep = list(range(1,PPO_EPOCHS+1))
for ax,(k,t,c) in zip(axes.flat, [('reward','Avg Reward','b'),('match','Match Rate','g'),
                                    ('nli','NLI Score','r'),('loss','PPO Loss','m')]):
    ax.plot(ep, hist[k], f'{c}-o', lw=2); ax.set_title(t); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('lapsh_training.png',dpi=150); plt.show()

## Cell 14: Evaluation

In [ ]:
def eval_baseline(data, n=100):
    c=0
    for s in tqdm(data[:n], desc='Baseline'):
        ids = tokenizer(format_prompt(s['question']), return_tensors='pt',
                        truncation=True, max_length=256).input_ids.to(DEVICE)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
        c += answers_match(tokenizer.decode(out[0][ids.shape[1]:],skip_special_tokens=True), s['answer'])
    return c/n

def eval_caa(data, li, alpha, n=100):
    c=0; al=CANDIDATE_LAYERS[li]
    for s in tqdm(data[:n], desc=f'CAA L{al}'):
        ids = tokenizer(format_prompt(s['question']), return_tensors='pt',
                        truncation=True, max_length=256).input_ids.to(DEVICE)
        c += answers_match(generate_with_steering(ids,steering_vectors,al,alpha), s['answer'])
    return c/n

def eval_lapsh(data, n=100):
    c=0; lu=Counter(); av=[]
    policy.eval()
    for s in tqdm(data[:n], desc='LAPSH'):
        st = extract_state(s['question']).unsqueeze(0)
        with torch.no_grad():
            lg,al = policy(st)
            li=lg.argmax(-1).item(); a=al[0].item()
        layer=CANDIDATE_LAYERS[li]; lu[layer]+=1; av.append(a)
        ids = tokenizer(format_prompt(s['question']), return_tensors='pt',
                        truncation=True, max_length=256).input_ids.to(DEVICE)
        c += answers_match(generate_with_steering(ids,steering_vectors,layer,a), s['answer'])
    return c/n, dict(lu), av

N=100
base_acc             = eval_baseline(eval_data, N)
caa_acc              = eval_caa(eval_data, li=2, alpha=10.0, n=N)
lapsh_acc, ldist, avs = eval_lapsh(eval_data, N)

print(f"Baseline:  {base_acc*100:.1f}%")
print(f"CAA fixed: {caa_acc*100:.1f}%")
print(f"LAPSH:     {lapsh_acc*100:.1f}%  (+{(lapsh_acc-base_acc)*100:.1f} pp)")
print(f"Layer dist: {ldist} | Mean alpha: {np.mean(avs):.2f}")

## Cell 15: Interpretability

In [ ]:
preferred = max(ldist, key=ldist.get)
ls = sorted(ldist.keys()); cs = [ldist.get(l,0) for l in ls]

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('LAPSH: Policy Behavior', fontsize=13, fontweight='bold')
bars = axes[0].bar([str(l) for l in ls], cs,
    color=['#2196F3' if c==max(cs) else '#90CAF9' for c in cs])
axes[0].set_title('Layer Preference'); axes[0].bar_label(bars); axes[0].grid(alpha=0.3,axis='y')
axes[1].hist(avs,bins=20,color='#4CAF50',edgecolor='white',alpha=0.8)
axes[1].axvline(np.mean(avs),color='red',linestyle='--',label=f'Mean={np.mean(avs):.2f}')
axes[1].set_title('Alpha Distribution'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('lapsh_interp.png',dpi=150); plt.show()
print(f"Preferred layer: {preferred}")

## Cell 16: Per-Layer Ablation

In [ ]:
print("Per-layer ablation (n=50)...")
ablation = {}
for i,l in enumerate(CANDIDATE_LAYERS):
    acc = eval_caa(eval_data, li=i, alpha=float(np.mean(avs)), n=50)
    ablation[l] = acc; print(f"  Layer {l}: {acc*100:.1f}%")
best_abl = max(ablation, key=ablation.get)
print(f"Best by ablation: {best_abl} | Policy preferred: {preferred} | Match: {best_abl==preferred}")

plt.figure(figsize=(8,4))
plt.bar([str(l) for l in ablation],[v*100 for v in ablation.values()],color='#FF7043',alpha=0.85)
plt.axhline(base_acc*100,color='gray',linestyle='--',label=f'Baseline {base_acc*100:.1f}%')
plt.axhline(lapsh_acc*100,color='blue',linestyle='--',label=f'LAPSH {lapsh_acc*100:.1f}%')
plt.title('Per-Layer Ablation'); plt.xlabel('Layer'); plt.ylabel('Acc (%)'); plt.legend()
plt.grid(alpha=0.3,axis='y'); plt.tight_layout()
plt.savefig('lapsh_ablation.png',dpi=150); plt.show()

## Cell 17: Save

In [ ]:
torch.save({
    'policy': policy.state_dict(), 'value': value_net.state_dict(),
    'steering_vectors': steering_vectors,
    'config': {'CANDIDATE_LAYERS':CANDIDATE_LAYERS,'HIDDEN_SIZE':HIDDEN_SIZE,'MODEL_ID':MODEL_ID},
    'results': {'baseline':base_acc,'caa':caa_acc,'lapsh':lapsh_acc,
                'layer_dist':ldist,'ablation':ablation,'mean_alpha':float(np.mean(avs))}
}, 'lapsh_checkpoint.pt')
print("Saved: lapsh_checkpoint.pt")